In [2]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from urllib.parse import quote

# --- 設定・関数定義エリア ---

def parse_price(price_str):
    """「X.X万円」という文字列を円単位の数値に変換する"""
    try:
        num_part = re.search(r'[\d.]+', price_str).group()
        return float(num_part) * 10000
    except (AttributeError, ValueError):
        return 0

def calculate_burden(rent, admin_fee, parking_fee=30000):
    """JTB様の社宅規程に基づいて自己負担額を計算する"""
    rent_cap = 110000
    user_rent_burden = 0

    if rent <= rent_cap:
        user_rent_burden = rent * 0.25
    else:
        overage = rent - rent_cap
        user_rent_burden = (rent_cap * 0.25) + overage

    total_burden = user_rent_burden + admin_fee + parking_fee
    return int(total_burden)

# =================================================================
# ▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼【ここからが改良部分】▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼▼
# =================================================================

# SUUMO用のエリアコード一覧 (首都圏対応版)
AREA_MASTER_CODES = {
    "東京都": {
        "千代田区": "13101", "中央区": "13102", "港区": "13103",
        "新宿区": "13104", "文京区": "13105", "台東区": "13106",
        "墨田区": "13107", "江東区": "13108", "品川区": "13109",
        "目黒区": "13110", "大田区": "13111", "世田谷区": "13112",
        "渋谷区": "13113", "中野区": "13114", "杉並区": "13115",
        "豊島区": "13116", "北区": "13117", "荒川区": "13118",
        "板橋区": "13119", "練馬区": "13120", "足立区": "13121",
        "葛飾区": "13122", "江戸川区": "13123",
    },
    "神奈川県": {
        "横浜市鶴見区": "14101", "横浜市神奈川区": "14102", "横浜市西区": "14103",
        "横浜市中区": "14104", "横浜市南区": "14105", "横浜市港北区": "14109",
        "川崎市川崎区": "14131", "川崎市幸区": "14132", "川崎市中原区": "14133",
        "川崎市高津区": "14134", "川崎市多摩区": "14135", "藤沢市": "14205",
        "鎌倉市": "14204", "横須賀市": "14201",
    },
    "千葉県": {
        "千葉市中央区": "12101", "市川市": "12203", "船橋市": "12204",
        "松戸市": "12207", "習志野市": "12216", "浦安市": "12227",
        "柏市": "12217",
    }
}

# --- ユーザー入力部分 ---
print("■ ご希望の検索条件を入力してください。")

# 1. 都道府県の選択
print(f"選択可能な都道府県: {', '.join(AREA_MASTER_CODES.keys())}")
input_pref = input("都道府県名 (例: 神奈川県) > ")

if input_pref not in AREA_MASTER_CODES:
    print("エラー: 選択可能な都道府県名を入力してください。")
    exit()

# 2. 市区町村の選択
available_cities = AREA_MASTER_CODES[input_pref]
print(f"--- {input_pref}で選択可能な市区町村 ---")
print(f"{', '.join(available_cities.keys())}")
input_city = input("市区町村名 (例: 横浜市西区) > ")

if input_city not in available_cities:
    print("エラー: 選択可能な市区町村名を入力してください。")
    exit()

# 3. 追加キーワードの入力
input_keywords = input("追加の希望条件 (例: 駐車場 カウンターキッチン) > ")

# --- URLを動的に生成 ---
area_code = available_cities[input_city]
encoded_keywords = quote(input_keywords)

base_url = "https://suumo.jp/jj/chintai/ichiran/FR301FC001/?"
params = {
    "ar": "030", "bs": "040", "pc": "50", "smk": "", "po1": "25", "po2": "99",
    "shkr1": "03", "shkr2": "03", "shkr3": "03", "shkr4": "03",
    "sc": area_code,
    "ta": "13", "cb": "0.0", "ct": "15.0", "md": "05", "md": "06", "md": "07", "md": "08",
    "ts": "1", "et": "9999999", "mb": "0", "mt": "55", "cn": "15",
    "fw2": encoded_keywords,
    "kr": "A"
}
# 神奈川(14)、千葉(12)の場合はta(都道府県コード)を変更
if input_pref == "神奈川県":
    params["ta"] = "14"
elif input_pref == "千葉県":
    params["ta"] = "12"

target_url = base_url + "&".join([f"{k}={v}" for k, v in params.items()])

print("\n--- 検索を開始します ---")
print(f"検索エリア: {input_pref} {input_city}")
print(f"追加キーワード: {input_keywords}")
print(f"生成されたURL: {target_url}\n")

# =================================================================
# ▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲【ここまでが改良部分】▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲▲
# =================================================================


# --- スクレイピングと分析処理 ---
headers = {"User-Agent": "Mozilla/5.0"}
response = requests.get(target_url, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

property_list = []
properties = soup.find_all('div', class_='cassetteitem')

if not properties:
    print("指定された条件に一致する物件が見つかりませんでした。")
else:
    for prop in properties:
        try:
            name = prop.find('div', class_='cassetteitem_content-title').text.strip()
            address = prop.find('li', class_='cassetteitem_detail-col1').text.strip()
            rent_text = prop.find('span', class_='cassetteitem_price--rent').text
            rent_val = parse_price(rent_text)
            admin_fee_text = prop.find('span', class_='cassetteitem_price--administration').text
            admin_fee_val = parse_price(admin_fee_text)

            property_list.append({
                "物件名": name, "住所": address, "賃料": rent_val, "管理費": admin_fee_val,
            })
        except Exception as e:
            continue

    df = pd.DataFrame(property_list)
    df = df[df["賃料"] > 0].reset_index(drop=True)

    if df.empty:
        print("データが取得できませんでした。サイトの構造が変更された可能性があります。")
    else:
        df["自己負担額"] = df.apply(lambda row: calculate_burden(row["賃料"], row["管理費"]), axis=1)
        average_burden = df["自己負担額"].mean()
        df["平均との差額"] = df["自己負担額"] - average_burden

        print(f"■ 分析結果：自己負担額の平均は {average_burden:,.0f} 円です。\n")

        df_display = df[["物件名", "賃料", "管理費", "自己負担額", "平均との差額"]].copy()
        df_display["賃料"] = df_display["賃料"].map('{:,.0f}円'.format)
        df_display["管理費"] = df_display["管理費"].map('{:,.0f}円'.format)
        df_display["自己負担額"] = df_display["自己負担額"].map('{:,.0f}円'.format)
        df_display["平均との差額"] = df_display["平均との差額"].map('{:,.0f}円'.format)

        print("■ 物件リストと費用分析")
        print(df_display.to_string())

■ ご希望の検索条件を入力してください。
選択可能な都道府県: 東京都, 神奈川県, 千葉県
--- 千葉県で選択可能な市区町村 ---
千葉市中央区, 市川市, 船橋市, 松戸市, 習志野市, 浦安市, 柏市

--- 検索を開始します ---
検索エリア: 千葉県 松戸市
追加キーワード: 
生成されたURL: https://suumo.jp/jj/chintai/ichiran/FR301FC001/?ar=030&bs=040&pc=50&smk=&po1=25&po2=99&shkr1=03&shkr2=03&shkr3=03&shkr4=03&sc=12207&ta=12&cb=0.0&ct=15.0&md=08&ts=1&et=9999999&mb=0&mt=55&cn=15&fw2=&kr=A

指定された条件に一致する物件が見つかりませんでした。
